# Reviewed Gemma 3 4B: controlled four-epoch pilot
Import as a **new private notebook**. Select **GPU T4 x2**, internet, and the enabled **HF_TOKEN** Secret. Attach (1) the reviewed Dataset `fama_jensen_1983_pilot_reviewed.jsonl`, (2) **version 2 output of `oliverlds/compare-few-shot`**, and (3) **version 1 output of `oliverlds/smoke-4b-reviewed`**. This starts a **fresh** rank-8 adapter on all 48 reviewed training pairs, with the frozen explicit instruction and three demonstrations. It uses **one T4**, NF4 weights, FP32 compute, learning rate `5e-5`, batch size 1, accumulation 2, and four epochs (96 optimizer updates). No smoke adapter is loaded. All 12 validation examples are evaluated and generated before training and after every epoch (60 rewrites total); generation can take longer than training. Twelve reserved test rows remain unused. Epoch adapters, incremental `evaluation.json`, `claim_review.md`, label audit, training log, and `pilot_status.json` are saved under `/kaggle/working/reviewed_4b_pilot/`. Claim checkboxes require review; loss and copying flags alone do not establish faithfulness. Run the final cell once per fresh session; it rejects a nonempty output directory.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0] or "0"


In [ ]:
%pip install -q -U "transformers==5.20.1" "peft==0.21.2" "accelerate>=1.0" "datasets>=3.0" "bitsandbytes>=0.46" "sentencepiece" "Pillow"


In [ ]:
%%writefile /kaggle/working/compare_few_shot.py
"""Frozen prompt formatting used by historical experiments."""

INSTRUCTION = (
    "Rewrite the passage in the academic economics style of Eugene Fama and his "
    "coauthored Fama-Jensen writing, as demonstrated by the examples below. "
    "Use precise economic terminology, direct analytical statements, explicit causal "
    "or conditional relationships where present, and concise definitions. Preserve "
    "every substantive claim, qualification, uncertainty, negation, and attribution. "
    "Preserve who acts and who bears risk. Use specialized terms only when they "
    "express the same meaning. Do not import facts, names, examples, citations, or "
    "claims from the demonstrations. Do not add claims or strengthen tentative "
    "claims. Return only one coherent rewritten paragraph, without headings, "
    "Markdown, explanation, or commentary."
)

def make_prompt(demos, source):
    parts = [INSTRUCTION]
    for i, demo in enumerate(demos, 1):
        parts.append(f"Example {i}\nInput:\n{demo['source']}\nRewrite:\n{demo['target']}")
    parts.append(f"Now rewrite this passage:\n{source}\n\nRewrite:")
    return "\n\n".join(parts)


"""Inference-only comparison of Gemma 3 1B, 4B, and 12B with reviewed examples."""

import argparse
import hashlib
import json
import os
import re
from pathlib import Path


MODEL_IDS = ("google/gemma-3-1b-it", "google/gemma-3-4b-it", "google/gemma-3-12b-it")
DEMO_IDS = ("p014", "p018", "p075")
SOURCE_EDITS = {
    "p018": (("writing and enforcing agreements costs money",
              "writing and enforcing agreements is costly"),),
    "p075": (("The main aim is to explain why organizations survive when financial ownership and decision-making authority are separated.",
              "The main aim is to explain the factors that help organizations survive when financial ownership and decision-making authority are separated, a distinction often described imprecisely as ownership and control."),),
}


def normalized(text):
    return " ".join(text.split())


def validate_completion(report):
    if len(report["results"]) != 12 or any(not r["output"].strip() for r in report["results"]):
        raise RuntimeError("Comparison requires 12 nonempty generations; report remains incomplete")


def validate_baseline(path, dataset_path, demos, validation):
    report = json.loads(Path(path).read_text(encoding="utf-8"))
    validate_completion(report)
    if (report.get("model_id") != MODEL_IDS[1] or report.get("complete") is not True
            or report.get("training_called") is not False or report.get("test_rows_used") != 0
            or report.get("quantization") != "NF4 double quantization, FP32 compute"
            or report.get("dataset_sha256") != hashlib.sha256(Path(dataset_path).read_bytes()).hexdigest()
            or report.get("instruction") != INSTRUCTION or report.get("demonstrations") != demos
            or report.get("generation") != {"do_sample": False, "max_new_tokens": 512}):
        raise ValueError("Attach the completed FP32 4B report from compare-few-shot version 2")
    for old, row in zip(report["results"], validation):
        prompt = make_prompt(demos, row["plain_english"])
        if (old.get("source_ids") != row["source_ids"]
                or old.get("source") != row["plain_english"]
                or old.get("reference") != row["completion"]
                or old.get("prompt_sha256") != hashlib.sha256(prompt.encode()).hexdigest()
                or old.get("hit_output_limit") is not False):
            raise ValueError("4B baseline prompts, sources, or generation validity do not match")
    return report


def validate_gpu_placement(device_map):
    if not device_map:
        raise RuntimeError("Missing model device map")
    devices = {str(value) for value in device_map.values()}
    if not devices <= {"0", "1", "cuda:0", "cuda:1"}:
        raise RuntimeError("12B must fit on both GPUs without CPU or disk offload")
    if {value.split(":")[-1] for value in devices} != {"0", "1"}:
        raise RuntimeError("Expected model layers on both GPUs; inspect device map")


def validate_probe(result):
    if not result["output"].strip() or result["hit_output_limit"]:
        raise RuntimeError("First-generation check failed: empty or output-limited text")


def load_experiment(path):
    rows = [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines()
            if line.strip()]
    groups = {split: [r for r in rows if r.get("split") == split]
              for split in ("train", "validation", "test")}
    if len(rows) != 78 or [len(groups[k]) for k in groups] != [54, 12, 12]:
        raise ValueError("Expected the instructed 78-pair Dataset with 54/12/12 splits")
    for row in rows:
        if row.get("prompt_format") != "rewrite_v1" or any(
                not isinstance(row.get(k), str) or not row[k].strip()
                for k in ("plain_english", "completion")):
            raise ValueError("Expected nonempty instructed source/target rows")
    demos = []
    for key in DEMO_IDS:
        matches = [row for row in groups["train"] if row.get("source_ids") == [key]]
        if len(matches) != 1:
            raise ValueError(f"Expected exactly one training demonstration {key}")
        row = matches[0]
        source = row["plain_english"]
        for old, new in SOURCE_EDITS.get(key, ()):
            if source.count(old) != 1:
                raise ValueError(f"Reviewed source text changed for {key}; review again")
            source = source.replace(old, new)
        demos.append({"source_id": key, "source": source,
                      "original_source": row["plain_english"],
                      "target": normalized(row["completion"])})
    sources = [normalized(row["plain_english"]) for row in groups["validation"]]
    if len(set(sources)) != 12 or any(normalized(d["source"]) in sources or
            normalized(d["original_source"]) in sources for d in demos):
        raise ValueError("Demonstrations and validation inputs must be disjoint")
    return demos, groups["validation"]




def load_hf_token():
    # Kaggle always retrieves the enabled Secret, never a pasted notebook value.
    if Path("/kaggle").exists():
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            raise RuntimeError("Enable the Kaggle Secret named HF_TOKEN") from None
        if not token:
            raise RuntimeError("The Kaggle HF_TOKEN Secret is empty")
        os.environ["HF_TOKEN"] = token
    return bool(os.environ.get("HF_TOKEN"))


def write_review(output_dir):
    paths = [output_dir / (model.split("/")[-1] + ".json") for model in MODEL_IDS]
    reports = [json.loads(p.read_text(encoding="utf-8")) for p in paths if p.exists()]
    if not reports:
        return
    lines = ["# Few-shot claim and style review", "",
             "Compare the same source across models. Sentence boundaries are review aids;",
             "check every qualification and causal relationship. All ratings remain blank.", "",
             "Style rubric (0 absent, 1 partial, 2 clear): precise economic vocabulary;",
             "analytical/conditional structure; concise prose resembling the demonstrations.",
             "Fluent or technical wording alone does not establish author style.", ""]
    for initial in reports[0]["results"]:
        key = initial["source_ids"]
        lines.extend([f"## {', '.join(key)}", "", "Source claims:", ""])
        claims = re.split(r"(?<=[.!?])\s+(?=[A-Z])", initial["source"])
        lines.extend(f"{i}. {claim}" for i, claim in enumerate(claims, 1))
        lines.extend(["", "Reference (not shown to either model):", "", initial["reference"], ""])
        for report in reports:
            matches = [r for r in report["results"] if r["source_ids"] == key]
            if not matches:
                continue
            row = matches[0]
            lines.extend([f"### {report['model_id']}", "", row["output"], ""])
            lines.extend(f"- Claim {i}: [ ] preserved [ ] changed [ ] omitted"
                         for i in range(1, len(claims) + 1))
            lines.extend(["- Unsupported additions: ", "- Qualifications or actors changed: ",
                          "- Economic vocabulary (0–2): ", "- Analytical structure (0–2): ",
                          "- Concision and demonstrated style (0–2): ", ""])
    (output_dir / "claim_style_review.md").write_text("\n".join(lines), encoding="utf-8")


def run_training(demos, validation, model_id, dataset_path, output_dir, access_only=False,
                 baseline=None):
    """Inference only; ML imports stay inside this GPU entry point."""
    dual_gpu = model_id == MODEL_IDS[2]
    if not dual_gpu:
        os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0] or "0"
    import torch
    import transformers
    from transformers import (AutoModelForCausalLM, AutoProcessor, AutoTokenizer,
                              BitsAndBytesConfig, Gemma3ForConditionalGeneration,
                              LogitsProcessor, LogitsProcessorList)

    if not torch.cuda.is_available():
        raise RuntimeError("Select a Kaggle T4 GPU")
    if dual_gpu and torch.cuda.device_count() != 2:
        raise RuntimeError("Select GPU T4 x2 and restart the session: exactly two visible GPUs are required")
    if not load_hf_token():
        raise RuntimeError("Set HF_TOKEN outside the repository")
    if access_only:
        from huggingface_hub import hf_hub_download
        for name in ((model_id,) if dual_gpu else MODEL_IDS[:2]):
            hf_hub_download(name, "config.json")
            print(f"Confirmed model access: {name}", flush=True)
        return
    # Gemma 3 4B can overflow in FP16 on T4. Keep 4-bit weights but use
    # FP32 compute/unquantized layers for both models to keep this comparison consistent.
    kwargs = {"dtype": torch.float32, "device_map": {"": 0},
              "attn_implementation": "sdpa",
              "quantization_config": BitsAndBytesConfig(
                  load_in_4bit=True, bnb_4bit_quant_type="nf4",
                  bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)}
    if dual_gpu:
        output_dir.mkdir(parents=True, exist_ok=True)
        hardware_path = output_dir / "12b_hardware_preflight.json"
        hardware = {"model_id": model_id, "status": "checking_memory", "devices": [
            {"device": i, "name": torch.cuda.get_device_name(i),
             "free_bytes": torch.cuda.mem_get_info(i)[0],
             "total_bytes": torch.cuda.mem_get_info(i)[1]} for i in (0, 1)]}
        hardware_path.write_text(json.dumps(hardware, indent=2) + "\n", encoding="utf-8")
        # Reserve space for CUDA, activations, generation logits, and the KV cache.
        memory_limits = {}
        for index in (0, 1):
            free, total = torch.cuda.mem_get_info(index)
            budget = min(free - (3 if index == 0 else 2) * 1024**3,
                         total - 3 * 1024**3)
            if budget < 8 * 1024**3:
                raise RuntimeError("Insufficient free GPU memory; start a fresh T4 x2 session")
            memory_limits[index] = budget
        kwargs.update(device_map="balanced", max_memory={**memory_limits, "cpu": 0})
        hardware.update(status="loading_model", gpu_weight_budgets_bytes=memory_limits)
        hardware_path.write_text(json.dumps(hardware, indent=2) + "\n", encoding="utf-8")
        print("Loading 12B across two GPUs with reserved generation headroom", flush=True)
    multimodal = model_id != MODEL_IDS[0]
    try:
        if multimodal:
            processor = AutoProcessor.from_pretrained(model_id)
            tokenizer = processor.tokenizer
            model = Gemma3ForConditionalGeneration.from_pretrained(model_id, **kwargs)
        else:
            processor = AutoTokenizer.from_pretrained(model_id)
            tokenizer = processor
            model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
        if dual_gpu:
            validate_gpu_placement(model.hf_device_map)
    except Exception:
        if dual_gpu:
            hardware["status"] = "loading_or_placement_failed"
            hardware_path.write_text(json.dumps(hardware, indent=2) + "\n", encoding="utf-8")
        raise
    model.eval()
    if dual_gpu:
        hardware["status"] = "placement_passed"
        hardware_path.write_text(json.dumps(hardware, indent=2) + "\n", encoding="utf-8")
        print("Confirmed GPU-only model placement across devices 0 and 1", flush=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    if baseline is not None:
        (output_dir / "gemma-3-4b-it.json").write_text(
            json.dumps(baseline, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    report_path = output_dir / (model_id.split("/")[-1] + ".json")
    report = {"model_id": model_id, "complete": False, "training_called": False, "test_rows_used": 0,
              "validation_rows": 12, "demonstrations": demos, "instruction": INSTRUCTION,
              "dataset_sha256": hashlib.sha256(dataset_path.read_bytes()).hexdigest(),
              "transformers_version": transformers.__version__, "torch_version": torch.__version__,
              "model_revision": getattr(model.config, "_commit_hash", None),
              "quantization": "NF4 double quantization, FP32 compute",
              "inference_revision": "fp32_with_generation_validation_v2",
              "generation": {"do_sample": False, "max_new_tokens": 512}, "results": []}

    def memory_snapshot():
        result = []
        for index in range(2 if dual_gpu else 1):
            free, total = torch.cuda.mem_get_info(index)
            result.append({"device": index, "name": torch.cuda.get_device_name(index),
                           "free_bytes": free, "total_bytes": total,
                           "allocated_bytes": torch.cuda.memory_allocated(index),
                           "reserved_bytes": torch.cuda.memory_reserved(index),
                           "peak_allocated_bytes": torch.cuda.max_memory_allocated(index)})
        return result

    if dual_gpu:
        report.update(inference_revision="12b_dual_t4_fp32_v1",
                      device_map={key: str(value) for key, value in model.hf_device_map.items()},
                      gpu_weight_budgets_bytes=memory_limits,
                      memory_after_load=memory_snapshot(),
                      baseline_model_id=baseline["model_id"] if baseline else None)
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

    class FiniteScores(LogitsProcessor):
        def __call__(self, input_ids, scores):
            # Negative infinity may represent a deliberately suppressed token.
            invalid = (torch.isnan(scores).any() | torch.isposinf(scores).any()
                       | ~torch.isfinite(scores).any(dim=-1).all())
            if bool(invalid):
                raise RuntimeError("Invalid generation scores (NaN/positive infinity/no finite token). "
                                   "The comparison cannot be evaluated.")
            return scores

    for index, row in enumerate(validation, 1):
        prompt = make_prompt(demos, row["plain_english"])
        content = [{"type": "text", "text": prompt}] if multimodal else prompt
        encoded = processor.apply_chat_template(
            [{"role": "user", "content": content}], tokenize=True,
            add_generation_prompt=True, return_dict=True, return_tensors="pt")
        input_device = model.get_input_embeddings().weight.device
        inputs = {k: v.to(input_device) for k, v in encoded.items()}
        input_length = int(inputs["input_ids"].shape[-1])
        if input_length + 512 > 4096:
            raise ValueError("Prompt exceeds the experiment's 4096-token budget; no truncation allowed")
        try:
            with torch.inference_mode():
                tokens = model.generate(**inputs, do_sample=False, max_new_tokens=512,
                                        use_cache=True, pad_token_id=tokenizer.pad_token_id,
                                        remove_invalid_values=False,
                                        logits_processor=LogitsProcessorList([FiniteScores()]))
        except RuntimeError:
            report["error"] = "Generation failed; consult notebook log. Report is incomplete."
            if dual_gpu:
                report["memory_at_generation_error"] = memory_snapshot()
            report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
            raise
        new_tokens = tokens[0, input_length:]
        output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        eos = model.generation_config.eos_token_id
        eos = eos if isinstance(eos, list) else [eos]
        report["results"].append({
            "source_ids": row["source_ids"], "source": row["plain_english"],
            "reference": row["completion"], "output": output,
            "prompt": prompt, "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
            "input_tokens": input_length, "output_tokens": int(new_tokens.numel()),
            "first_generated_token_ids": new_tokens[:16].tolist(),
            "hit_output_limit": new_tokens.numel() == 512 and int(new_tokens[-1]) not in eos,
            "copied_source": normalized(output) == normalized(row["plain_english"]),
            "claim_review_status": "pending_manual_review"})
        report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        if not output:
            raise RuntimeError("Generation decoded to empty text; stopped before processing more examples. "
                               "Report is incomplete and includes token IDs for diagnosis.")
        if dual_gpu and index == 1:
            report["first_generation_check"] = {
                "passed": False, "source_ids": row["source_ids"], "memory": memory_snapshot()}
            report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
            validate_probe(report["results"][-1])
            report["first_generation_check"]["passed"] = True
            report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
            print("First-generation check passed; continuing with the remaining 11 validation inputs", flush=True)
        write_review(output_dir)
        print(f"{model_id}: saved validation generation {index}/12", flush=True)
    validate_completion(report)
    report["complete"] = True
    if dual_gpu:
        report["memory_after_all_generations"] = memory_snapshot()
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"Saved report to {report_path}", flush=True)


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", required=True)
    parser.add_argument("--model-id", choices=MODEL_IDS, default=MODEL_IDS[0])
    parser.add_argument("--output-dir", default="/kaggle/working/few_shot_comparison")
    parser.add_argument("--check", action="store_true")
    parser.add_argument("--baseline", help="completed FP32 4B comparison JSON (required for 12B)")
    parser.add_argument("--check-access", action="store_true",
                        help="check GPU, Secret, and both model licenses before downloading weights")
    args = parser.parse_args(argv)
    dataset_path = Path(args.dataset)
    demos, validation = load_experiment(dataset_path)
    baseline = None
    if args.model_id == MODEL_IDS[2]:
        if not args.baseline:
            parser.error("12B requires --baseline pointing to the saved 4B version 2 JSON")
        baseline = validate_baseline(args.baseline, dataset_path, demos, validation)
    print("Validated 3 reviewed training demonstrations and 12 validation inputs; test rows excluded")
    if args.check:
        print("Dry-run OK. No model downloaded or trained.")
        return
    run_training(demos, validation, args.model_id, dataset_path, Path(args.output_dir), args.check_access,
                 baseline)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/smoke_4b_reviewed.py
"""Five-step 4B QLoRA pipeline/memory smoke test on semantically reviewed pairs."""

import argparse
import hashlib
import json
import math
import os
import re
from pathlib import Path

from compare_few_shot import DEMO_IDS, INSTRUCTION, load_hf_token, make_prompt, normalized, validate_completion

MODEL_ID = "google/gemma-3-4b-it"
FORMAT = "fama_style_few_shot_reviewed_v1"
PARENT_SHA = "3fabac4ddfedb947c88bdbb3956f21f7d524f1f51504fbca6067efffd272a8ed"
SMOKE_ANCHORS = ("p031", "p060")
EVAL_IDS = ("p008", "p050")
STEPS = 5
MAX_LENGTH = 2048
TARGET_PATTERN = r".*language_model\.layers\.\d+\.self_attn\.(q_proj|v_proj)$"


def choose_smoke_ids(lengths):
    if len(lengths) < 3 or any(key not in lengths for key in SMOKE_ANCHORS):
        raise ValueError("Smoke selection needs at least three training rows and both anchors")
    longest = max(lengths, key=lengths.get)
    candidates = [longest, *SMOKE_ANCHORS, *sorted(lengths, key=lengths.get, reverse=True)]
    return tuple(dict.fromkeys(candidates))[:3]


def load_inputs(dataset_path, baseline_path):
    rows = [json.loads(line) for line in Path(dataset_path).read_text(encoding="utf-8").splitlines() if line.strip()]
    groups = {key: [r for r in rows if r.get("split") == key] for key in ("train", "validation", "test")}
    if len(rows) != 72 or [len(groups[k]) for k in groups] != [48, 12, 12]:
        raise ValueError("Expected reviewed Dataset: 48 train, 12 validation, 12 reserved test rows")
    baseline = json.loads(Path(baseline_path).read_text(encoding="utf-8"))
    validate_completion(baseline)
    if (baseline.get("model_id") != MODEL_ID or baseline.get("complete") is not True
            or baseline.get("training_called") is not False or baseline.get("test_rows_used") != 0
            or baseline.get("dataset_sha256") != PARENT_SHA
            or baseline.get("quantization") != "NF4 double quantization, FP32 compute"
            or baseline.get("generation") != {"do_sample": False, "max_new_tokens": 512}
            or baseline.get("instruction") != INSTRUCTION
            or [d.get("source_id") for d in baseline.get("demonstrations", [])] != list(DEMO_IDS)):
        raise ValueError("Attach the completed FP32 4B report from compare-few-shot version 2")
    demos = baseline["demonstrations"]
    for row in groups["train"] + groups["validation"]:
        if (row.get("prompt_format") != FORMAT or row.get("parent_dataset_sha256") != PARENT_SHA
                or not isinstance(row.get("completion"), str) or not row["completion"].strip()
                or row.get("prompt") != make_prompt(demos, row["plain_english"])):
            raise ValueError("Dataset prompts must use the reviewed format and frozen baseline demonstrations")
    train_ids = [tuple(r["source_ids"]) for r in groups["train"]]
    if (len(set(train_ids)) != 48 or any((key,) in train_ids for key in DEMO_IDS)
            or any((key,) not in train_ids for key in SMOKE_ANCHORS)):
        raise ValueError("Training IDs must be unique, include smoke rows, and exclude demonstrations")
    for row, previous in zip(groups["validation"], baseline["results"]):
        if (row["source_ids"] != previous["source_ids"] or row["plain_english"] != previous["source"]
                or row["completion"] != previous["reference"] or previous.get("hit_output_limit")
                or hashlib.sha256(row["prompt"].encode()).hexdigest() != previous["prompt_sha256"]):
            raise ValueError("Validation sources, targets, and prompt hashes must match the saved baseline")
    if any(sum(r["source_ids"] == [key] for r in groups["validation"]) != 1 for key in EVAL_IDS):
        raise ValueError("Missing a fixed smoke validation row")
    return groups, baseline


def run_training(groups, baseline, dataset_path, output_dir):
    # Single-device training avoids unsupported inference-style multi-GPU dispatch.
    os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0] or "0"
    import torch
    import transformers
    import peft
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import (AutoProcessor, BitsAndBytesConfig, Gemma3ForConditionalGeneration,
                              LogitsProcessor, LogitsProcessorList, Trainer, TrainerCallback, TrainingArguments)

    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError("The training process needs one visible CUDA GPU; select Kaggle T4 x2")
    if not load_hf_token():
        raise RuntimeError("Enable HF_TOKEN in Kaggle Secrets")
    output_dir.mkdir(parents=True, exist_ok=True)
    status_path = output_dir / "smoke_status.json"
    status = {"model_id": MODEL_ID, "complete": False, "purpose": "pipeline and memory smoke test",
              "trained_ids": [], "selection": "longest training row plus two anchors; three unique rows",
              "validation_ids": list(EVAL_IDS), "optimizer_steps_planned": STEPS,
              "eligible_training_rows": 48, "test_rows_used": 0,
              "training_gpus_used": 1, "second_t4_used": False,
              "learning_rate": 5e-5, "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
              "target_modules_regex": TARGET_PATTERN, "max_length": MAX_LENGTH,
              "gradient_accumulation_steps": 2, "seed": 42,
              "dtype": "NF4 weights, FP32 compute and unquantized layers",
              "dataset_sha256": hashlib.sha256(Path(dataset_path).read_bytes()).hexdigest(),
              "transformers_version": transformers.__version__, "peft_version": peft.__version__,
              "torch_version": torch.__version__, "phase": "loading", "gradient_checks": []}

    def save_status():
        free, total = torch.cuda.mem_get_info(0)
        status["memory"] = {"gpu_name": torch.cuda.get_device_name(0), "free_bytes": free,
                            "total_bytes": total, "allocated_bytes": torch.cuda.memory_allocated(0),
                            "reserved_bytes": torch.cuda.memory_reserved(0),
                            "peak_allocated_bytes": torch.cuda.max_memory_allocated(0)}
        status_path.write_text(json.dumps(status, indent=2) + "\n", encoding="utf-8")

    save_status()
    torch.manual_seed(42)
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    tokenizer = processor.tokenizer
    model = Gemma3ForConditionalGeneration.from_pretrained(
        MODEL_ID, dtype=torch.float32, device_map={"": 0}, attn_implementation="sdpa",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32))
    model.config.use_cache = False
    model.config.text_config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                           gradient_checkpointing_kwargs={"use_reentrant": False})
    matched = [name for name, _ in model.named_modules() if re.fullmatch(TARGET_PATTERN, name)]
    if len(matched) != 2 * model.config.text_config.num_hidden_layers:
        raise RuntimeError("Expected q/v adapters only in every language layer; module layout changed")
    model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=TARGET_PATTERN, bias="none", task_type="CAUSAL_LM"))
    parameters = {name: value for name, value in model.named_parameters() if value.requires_grad}
    if not parameters or any("lora_" not in name or "language_model" not in name for name in parameters):
        raise RuntimeError("Only language LoRA parameters may train; base and vision weights must be frozen")
    before = {name: value.detach().cpu().clone() for name, value in parameters.items()}
    status.update(phase="model_loaded", trainable_parameters=sum(p.numel() for p in parameters.values()),
                  adapted_language_modules=len(matched), model_revision=getattr(model.config, "_commit_hash", None))
    save_status()

    def encode(row):
        user = {"role": "user", "content": [{"type": "text", "text": row["prompt"]}]}
        assistant = {"role": "assistant", "content": [{"type": "text", "text": row["completion"]}]}
        prefix = processor.apply_chat_template([user], tokenize=True, add_generation_prompt=True,
                                              return_dict=True, return_tensors="pt")["input_ids"][0].tolist()
        full = processor.apply_chat_template([user, assistant], tokenize=True,
                                            return_dict=True, return_tensors="pt")
        ids = full["input_ids"][0].tolist()
        if ids[:len(prefix)] != prefix or len(ids) <= len(prefix) or len(ids) > MAX_LENGTH:
            raise ValueError(f"Bad token boundary or overlength row {row['source_ids']}; no truncation permitted")
        labels = [-100] * len(prefix) + ids[len(prefix):]
        decoded = tokenizer.decode(ids[len(prefix):], skip_special_tokens=True)
        if normalized(decoded) != normalized(row["completion"]):
            raise ValueError(f"Supervised labels do not exactly match target for {row['source_ids']}")
        prepared = {key: value[0].tolist() for key, value in full.items()
                    if key in ("input_ids", "attention_mask", "token_type_ids")}
        prepared["labels"] = labels
        audit = {"source_ids": row["source_ids"], "split": row["split"], "full_tokens": len(ids),
                 "masked_prompt_tokens": len(prefix), "supervised_tokens": len(ids)-len(prefix),
                 "labels_match_target": True}
        return prepared, audit

    encoded, audits = {}, []
    for row in groups["train"] + groups["validation"]:
        prepared, audit = encode(row)
        encoded[tuple(row["source_ids"])] = prepared
        audits.append(audit)
    (output_dir / "prepared_label_audit.json").write_text(json.dumps(audits, indent=2) + "\n", encoding="utf-8")
    lengths = {a["source_ids"][0]: a["full_tokens"] for a in audits if a["split"] == "train"}
    smoke_ids = choose_smoke_ids(lengths)
    status.update(phase="labels_checked", max_full_tokens=max(a["full_tokens"] for a in audits),
                  trained_ids=list(smoke_ids), longest_training_tokens=max(lengths.values()))
    save_status()
    print("Verified completion-only labels and lengths for all 48 train and 12 validation rows", flush=True)

    def collate(features):
        if len(features) != 1:
            raise ValueError("This smoke test uses batch size one")
        return {key: torch.tensor([value], dtype=torch.long) for key, value in features[0].items()}

    class GradientChecks(TrainerCallback):
        def on_pre_optimizer_step(self, args, state, control, **kwargs):
            grads = [p.grad.detach() for p in parameters.values() if p.grad is not None]
            if not grads or any(not bool(torch.isfinite(g).all()) for g in grads):
                raise RuntimeError("Missing or non-finite adapter gradients")
            norm_squared = sum(float(g.float().square().sum().item()) for g in grads)
            if not math.isfinite(norm_squared) or norm_squared <= 0:
                raise RuntimeError("Adapter gradient norm must be finite and positive")
            status["gradient_checks"].append({"step": state.global_step+1, "finite": True,
                                               "norm": math.sqrt(norm_squared)})
            save_status()
            return control

        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs and "loss" in logs and not math.isfinite(logs["loss"]):
                raise RuntimeError("Non-finite training loss")

    args = TrainingArguments(output_dir=str(output_dir / "trainer"), max_steps=STEPS,
        per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=2,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate=5e-5, optim="adamw_torch", fp16=False, bf16=False,
        logging_steps=1, save_strategy="no", report_to="none", seed=42,
        remove_unused_columns=False, prediction_loss_only=True, label_names=["labels"])
    class FiniteLossTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            result = super().compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)
            loss = result[0] if return_outputs else result
            if not bool(torch.isfinite(loss).all()):
                raise RuntimeError("Non-finite loss during model forward pass")
            return result

    trainer = FiniteLossTrainer(model=model, args=args, processing_class=processor, data_collator=collate,
        train_dataset=Dataset.from_list([encoded[(key,)] for key in smoke_ids]),
        eval_dataset=Dataset.from_list([encoded[(key,)] for key in EVAL_IDS]),
        callbacks=[GradientChecks()])
    trainer.model_accepts_loss_kwargs = False
    with model.disable_adapter():
        original_loss = trainer.evaluate()["eval_loss"]
    if not math.isfinite(original_loss):
        raise RuntimeError("Base validation loss is non-finite")
    status["phase"] = "training"
    save_status()
    trainer.train()
    adapted_loss = trainer.evaluate()["eval_loss"]
    changed = any(not torch.equal(before[name], value.detach().cpu()) for name, value in parameters.items())
    if (trainer.state.global_step != STEPS or len(status["gradient_checks"]) != STEPS
            or not changed or not math.isfinite(adapted_loss)):
        raise RuntimeError("Smoke test did not complete five finite optimizer updates with changed adapter weights")
    status.update(phase="training_finished", optimizer_steps_completed=STEPS, adapter_weights_changed=True)
    save_status()
    model.save_pretrained(str(output_dir / "adapter"))
    processor.save_pretrained(str(output_dir / "adapter"))
    (output_dir / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2) + "\n")

    class FiniteScores(LogitsProcessor):
        def __call__(self, input_ids, scores):
            if bool(torch.isnan(scores).any() | torch.isposinf(scores).any()
                    | ~torch.isfinite(scores).any(dim=-1).all()):
                raise RuntimeError("Non-finite generation scores after training")
            return scores

    model.eval()
    generations = []
    for key in EVAL_IDS:
        row = next(r for r in groups["validation"] if r["source_ids"] == [key])
        old = next(r for r in baseline["results"] if r["source_ids"] == [key])
        inputs = processor.apply_chat_template([{"role": "user", "content": [
            {"type": "text", "text": row["prompt"]}]}], tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt")
        inputs = {name: value.to("cuda:0") for name, value in inputs.items()}
        with torch.inference_mode():
            tokens = model.generate(**inputs, max_new_tokens=512, do_sample=False, use_cache=True,
                pad_token_id=tokenizer.pad_token_id, remove_invalid_values=False,
                logits_processor=LogitsProcessorList([FiniteScores()]))
        generated = tokens[0, inputs["input_ids"].shape[-1]:]
        output = tokenizer.decode(generated, skip_special_tokens=True).strip()
        if not output:
            raise RuntimeError("Adapted generation is empty")
        generations.append({"source_ids": [key], "source": row["plain_english"],
                            "reference": row["completion"], "baseline_output": old["output"],
                            "smoke_adapter_output": output, "output_tokens": int(generated.numel()),
                            "at_output_budget": generated.numel() == 512,
                            "copied_source": normalized(output) == normalized(row["plain_english"])})
    evaluation = {"model_id": MODEL_ID, "purpose": "pipeline smoke test, not a style experiment",
                  "prompt_format": FORMAT, "train_rows": 3, "validation_rows": 2, "optimizer_steps": STEPS,
                  "original_validation_loss": original_loss, "fine_tuned_validation_loss": adapted_loss,
                  "adapter_weights_changed": True, "test_rows_used": 0, "generations": generations}
    (output_dir / "evaluation.json").write_text(json.dumps(evaluation, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    status.update(phase="complete", complete=True)
    save_status()
    print(f"Smoke complete: 3 train rows, 5 optimizer steps; saved {output_dir / 'evaluation.json'}", flush=True)


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", required=True)
    parser.add_argument("--baseline", required=True)
    parser.add_argument("--output-dir", default="/kaggle/working/gemma4b_reviewed_smoke")
    parser.add_argument("--check", action="store_true")
    args = parser.parse_args(argv)
    groups, baseline = load_inputs(args.dataset, args.baseline)
    print("Validated reviewed Dataset, demonstration exclusion, and identical baseline validation prompts")
    if args.check:
        print("Dry-run OK. No model downloaded or trained.")
        return
    output_dir = Path(args.output_dir)
    try:
        run_training(groups, baseline, args.dataset, output_dir)
    except Exception as exc:
        status_path = output_dir / "smoke_status.json"
        if status_path.exists():
            status = json.loads(status_path.read_text())
            status.update(complete=False, error_type=type(exc).__name__)
            status_path.write_text(json.dumps(status, indent=2) + "\n")
        raise


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/track_reviewed_4b.py
"""Controlled four-epoch 4B pilot with complete validation generations and claim review."""

import argparse
import hashlib
import json
import math
import os
import re
from pathlib import Path

from compare_few_shot import load_hf_token, normalized
from smoke_4b_reviewed import MODEL_ID, FORMAT, MAX_LENGTH, TARGET_PATTERN, load_inputs

EPOCHS = 4
TOTAL_STEPS = 96  # 48 rows / accumulation 2, batch 1, one GPU, four epochs.


def validate_smoke(path, dataset_path):
    status = json.loads(Path(path).read_text(encoding="utf-8"))
    if (status.get("complete") is not True or status.get("phase") != "complete"
            or status.get("model_id") != MODEL_ID or status.get("optimizer_steps_completed") != 5
            or status.get("adapter_weights_changed") is not True
            or status.get("dataset_sha256") != hashlib.sha256(Path(dataset_path).read_bytes()).hexdigest()
            or len(status.get("gradient_checks", [])) != 5
            or any(g.get("finite") is not True or not math.isfinite(g.get("norm", 0))
                   or g.get("norm", 0) <= 0 for g in status.get("gradient_checks", []))):
        raise ValueError("Attach the successful reviewed 4B smoke output for this exact Dataset")
    return status


def write_claim_review(report, path):
    lines = ["# Reviewed 4B pilot: validation claim review", "",
             "Use each source below to build a claim inventory before judging its rewrites.",
             "Check each source's actors, qualifications, causal direction and mechanisms.",
             "Boxes are deliberately blank; copying flags do not measure claim preservation.", ""]
    for row in report["validation_examples"]:
        source_id = row["source_ids"][0]
        lines += [f"## {source_id}", "", "### Source", "", row["source"], "",
                  "### Reference", "", row["reference"], ""]
        for snap in report["snapshots"]:
            result = next((r for r in snap["generations"] if r["source_ids"] == row["source_ids"]), None)
            if result is None:
                continue
            lines += [f"### Epoch {snap['epoch']} — step {snap['optimizer_step']}", "", result["output"], "",
                      "- [ ] All source claims retained", "- [ ] No unsupported claims added",
                      "- [ ] Actors and their roles retained", "- [ ] Qualifications retained",
                      "- [ ] Causal direction and mechanisms retained", "- [ ] One paragraph; no explanation or repetition",
                      "- Added/lost/changed claims (quote source and output):", "- Style and clarity notes:", ""]
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")


def validate_snapshots(report):
    expected = [r["source_ids"] for r in report["validation_examples"]]
    snapshots = report["snapshots"]
    if len(expected) != 12 or [s["epoch"] for s in snapshots] != list(range(EPOCHS+1)):
        raise ValueError("Expected baseline and four complete validation snapshots")
    for snap in snapshots:
        if (snap.get("complete") is not True or snap["optimizer_step"] != snap["epoch"]*24
                or not math.isfinite(snap["validation_loss"])
                or [r["source_ids"] for r in snap["losses"]] != expected
                or any(not math.isfinite(r["loss"]) for r in snap["losses"])
                or [r["source_ids"] for r in snap["generations"]] != expected
                or any(not r["output"].strip() for r in snap["generations"])):
            raise ValueError("Incomplete validation snapshot, invalid loss, or mismatched held-out IDs")


def run_training(groups, baseline, dataset_path, output_dir):
    # Single-device training avoids unsupported inference-style multi-GPU dispatch.
    os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0] or "0"
    import torch
    import transformers
    import peft
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import (AutoProcessor, BitsAndBytesConfig, Gemma3ForConditionalGeneration,
                              LogitsProcessor, LogitsProcessorList, Trainer, TrainerCallback, TrainingArguments)

    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError("The training process needs one visible CUDA GPU; select Kaggle T4 x2")
    if not load_hf_token():
        raise RuntimeError("Enable HF_TOKEN in Kaggle Secrets")
    output_dir.mkdir(parents=True, exist_ok=True)
    status_path = output_dir / "pilot_status.json"
    status = {"model_id": MODEL_ID, "complete": False, "purpose": "four-epoch reviewed-pair pilot",
              "trained_ids": [r["source_ids"][0] for r in groups["train"]],
              "validation_ids": [r["source_ids"][0] for r in groups["validation"]],
              "epochs_planned": EPOCHS, "optimizer_steps_planned": TOTAL_STEPS,
              "eligible_training_rows": 48, "test_rows_used": 0,
              "training_gpus_used": 1, "second_t4_used": False,
              "learning_rate": 5e-5, "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
              "target_modules_regex": TARGET_PATTERN, "max_length": MAX_LENGTH,
              "gradient_accumulation_steps": 2, "seed": 42,
              "dtype": "NF4 weights, FP32 compute and unquantized layers",
              "dataset_sha256": hashlib.sha256(Path(dataset_path).read_bytes()).hexdigest(),
              "transformers_version": transformers.__version__, "peft_version": peft.__version__,
              "torch_version": torch.__version__, "phase": "loading", "gradient_checks": []}

    def save_status():
        free, total = torch.cuda.mem_get_info(0)
        status["memory"] = {"gpu_name": torch.cuda.get_device_name(0), "free_bytes": free,
                            "total_bytes": total, "allocated_bytes": torch.cuda.memory_allocated(0),
                            "reserved_bytes": torch.cuda.memory_reserved(0),
                            "peak_allocated_bytes": torch.cuda.max_memory_allocated(0)}
        status_path.write_text(json.dumps(status, indent=2) + "\n", encoding="utf-8")

    save_status()
    torch.manual_seed(42)
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    tokenizer = processor.tokenizer
    model = Gemma3ForConditionalGeneration.from_pretrained(
        MODEL_ID, dtype=torch.float32, device_map={"": 0}, attn_implementation="sdpa",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32))
    model.config.use_cache = False
    model.config.text_config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                           gradient_checkpointing_kwargs={"use_reentrant": False})
    matched = [name for name, _ in model.named_modules() if re.fullmatch(TARGET_PATTERN, name)]
    if len(matched) != 2 * model.config.text_config.num_hidden_layers:
        raise RuntimeError("Expected q/v adapters only in every language layer; module layout changed")
    model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=TARGET_PATTERN, bias="none", task_type="CAUSAL_LM"))
    parameters = {name: value for name, value in model.named_parameters() if value.requires_grad}
    if not parameters or any("lora_" not in name or "language_model" not in name for name in parameters):
        raise RuntimeError("Only language LoRA parameters may train; base and vision weights must be frozen")
    before = {name: value.detach().cpu().clone() for name, value in parameters.items()}
    status.update(phase="model_loaded", trainable_parameters=sum(p.numel() for p in parameters.values()),
                  adapted_language_modules=len(matched), model_revision=getattr(model.config, "_commit_hash", None))
    save_status()

    def encode(row):
        user = {"role": "user", "content": [{"type": "text", "text": row["prompt"]}]}
        assistant = {"role": "assistant", "content": [{"type": "text", "text": row["completion"]}]}
        prefix = processor.apply_chat_template([user], tokenize=True, add_generation_prompt=True,
                                              return_dict=True, return_tensors="pt")["input_ids"][0].tolist()
        full = processor.apply_chat_template([user, assistant], tokenize=True,
                                            return_dict=True, return_tensors="pt")
        ids = full["input_ids"][0].tolist()
        if ids[:len(prefix)] != prefix or len(ids) <= len(prefix) or len(ids) > MAX_LENGTH:
            raise ValueError(f"Bad token boundary or overlength row {row['source_ids']}; no truncation permitted")
        labels = [-100] * len(prefix) + ids[len(prefix):]
        decoded = tokenizer.decode(ids[len(prefix):], skip_special_tokens=True)
        if normalized(decoded) != normalized(row["completion"]):
            raise ValueError(f"Supervised labels do not exactly match target for {row['source_ids']}")
        prepared = {key: value[0].tolist() for key, value in full.items()
                    if key in ("input_ids", "attention_mask", "token_type_ids")}
        prepared["labels"] = labels
        audit = {"source_ids": row["source_ids"], "split": row["split"], "full_tokens": len(ids),
                 "masked_prompt_tokens": len(prefix), "supervised_tokens": len(ids)-len(prefix),
                 "labels_match_target": True}
        return prepared, audit

    encoded, audits = {}, []
    for row in groups["train"] + groups["validation"]:
        prepared, audit = encode(row)
        encoded[tuple(row["source_ids"])] = prepared
        audits.append(audit)
    (output_dir / "prepared_label_audit.json").write_text(json.dumps(audits, indent=2) + "\n", encoding="utf-8")
    status.update(phase="labels_checked", max_full_tokens=max(a["full_tokens"] for a in audits))
    save_status()
    processor.save_pretrained(str(output_dir / "processor"))
    report = {"model_id": MODEL_ID, "complete": False, "prompt_format": FORMAT,
              "train_rows": 48, "validation_rows": 12, "test_rows_used": 0,
              "initialization": "fresh adapter; no smoke adapter loaded",
              "dataset_sha256": status["dataset_sha256"], "epochs_planned": EPOCHS,
              "optimizer_steps_planned": TOTAL_STEPS,
              "generation": {"do_sample": False, "max_new_tokens": 512},
              "validation_examples": [{"source_ids": r["source_ids"], "source": r["plain_english"],
                  "reference": r["completion"], "prompt_sha256": hashlib.sha256(r["prompt"].encode()).hexdigest()}
                  for r in groups["validation"]],
              "saved_untrained_baseline": baseline, "snapshots": []}

    def persist():
        (output_dir / "evaluation.json").write_text(json.dumps(report, indent=2, ensure_ascii=False)+"\n", encoding="utf-8")
        write_claim_review(report, output_dir / "claim_review.md")

    def collate(features):
        if len(features) != 1:
            raise ValueError("This pilot uses batch size one")
        return {key: torch.tensor([value], dtype=torch.long) for key, value in features[0].items()}

    class FiniteScores(LogitsProcessor):
        def __call__(self, input_ids, scores):
            if bool(torch.isnan(scores).any() | torch.isposinf(scores).any()
                    | ~torch.isfinite(scores).any(dim=-1).all()):
                raise RuntimeError("Non-finite generation scores")
            return scores

    def snapshot(epoch, step):
        # Called at baseline and from on_epoch_end; restore mode before Trainer continues.
        was_training = model.training
        model.eval()
        snap = {"epoch": epoch, "optimizer_step": step, "complete": False,
                "losses": [], "generations": []}
        report["snapshots"].append(snap)
        status.update(phase="validation", current_epoch=epoch, validation_generations_completed=0)
        save_status()
        persist()
        try:
            for row in groups["validation"]:
                batch = {name: value.to("cuda:0") for name, value in
                         collate([encoded[tuple(row["source_ids"])]]).items()}
                with torch.inference_mode():
                    loss = float(model(**batch).loss.item())
                if not math.isfinite(loss):
                    raise RuntimeError("Non-finite validation loss")
                snap["losses"].append({"source_ids": row["source_ids"], "loss": loss})
            # Same mean-per-example aggregation as the smoke Trainer evaluation.
            snap["validation_loss"] = sum(r["loss"] for r in snap["losses"])/len(snap["losses"])
            for row in groups["validation"]:
                inputs = processor.apply_chat_template([{"role": "user", "content": [
                    {"type": "text", "text": row["prompt"]}]}], tokenize=True,
                    add_generation_prompt=True, return_dict=True, return_tensors="pt")
                inputs = {name: value.to("cuda:0") for name, value in inputs.items()}
                with torch.inference_mode():
                    tokens = model.generate(**inputs, max_new_tokens=512, do_sample=False, use_cache=True,
                        pad_token_id=tokenizer.pad_token_id, remove_invalid_values=False,
                        logits_processor=LogitsProcessorList([FiniteScores()]))
                generated = tokens[0, inputs["input_ids"].shape[-1]:]
                output = tokenizer.decode(generated, skip_special_tokens=True).strip()
                if not output:
                    raise RuntimeError("Empty validation generation")
                old = next(r for r in baseline["results"] if r["source_ids"] == row["source_ids"])
                snap["generations"].append({"source_ids": row["source_ids"], "output": output,
                    "output_tokens": int(generated.numel()), "at_output_budget": generated.numel() == 512,
                    "copied_source": normalized(output) == normalized(row["plain_english"]),
                    "matches_reference": normalized(output) == normalized(row["completion"]),
                    "matches_saved_baseline": normalized(output) == normalized(old["output"]),
                    "claim_review": {"claims_added": None, "claims_lost": None,
                        "actors_changed": None, "qualifications_changed": None,
                        "causal_direction_changed": None, "notes": None}})
                status["validation_generations_completed"] = len(snap["generations"])
                persist()
                save_status()
                print(f"Epoch {epoch}: validation generation {len(snap['generations'])}/12 ({row['source_ids'][0]})", flush=True)
            snap["complete"] = True
            persist()
            print(f"Epoch {epoch} complete: validation loss {snap['validation_loss']:.6f}", flush=True)
        finally:
            model.train(was_training)
        status["phase"] = "training" if epoch < EPOCHS else "finalizing"
        save_status()

    class PilotChecks(TrainerCallback):
        def on_pre_optimizer_step(self, args, state, control, **kwargs):
            grads = [p.grad.detach() for p in parameters.values() if p.grad is not None]
            if not grads or any(not bool(torch.isfinite(g).all()) for g in grads):
                raise RuntimeError("Missing or non-finite adapter gradients")
            norm = math.sqrt(sum(float(g.float().square().sum().item()) for g in grads))
            if not math.isfinite(norm) or norm <= 0:
                raise RuntimeError("Adapter gradient norm must be finite and positive")
            status["gradient_checks"].append({"step": state.global_step+1, "finite": True, "norm": norm})
            save_status()

        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs and "loss" in logs and not math.isfinite(logs["loss"]):
                raise RuntimeError("Non-finite training loss")

        def on_epoch_end(self, args, state, control, **kwargs):
            epoch = round(state.epoch)
            if not math.isclose(state.epoch, epoch) or state.global_step != epoch*24:
                raise RuntimeError("Unexpected epoch/update boundary")
            status["optimizer_steps_completed"] = state.global_step
            # Preserve the epoch adapter even if a later generation fails.
            model.save_pretrained(str(output_dir / f"epoch_{epoch}" / "adapter"))
            snapshot(epoch, state.global_step)
            (output_dir / "training_log.json").write_text(json.dumps(state.log_history, indent=2)+"\n")
            return control

    class FiniteLossTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            result = super().compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)
            loss = result[0] if return_outputs else result
            if not bool(torch.isfinite(loss).all()):
                raise RuntimeError("Non-finite training forward loss")
            return result

    args = TrainingArguments(output_dir=str(output_dir / "trainer"), num_train_epochs=EPOCHS,
        per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=2,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate=5e-5, lr_scheduler_type="linear", warmup_steps=0, optim="adamw_torch",
        fp16=False, bf16=False, logging_steps=1, save_strategy="no", report_to="none", seed=42,
        remove_unused_columns=False, prediction_loss_only=True, label_names=["labels"])
    trainer = FiniteLossTrainer(model=model, args=args, processing_class=processor, data_collator=collate,
        train_dataset=Dataset.from_list([encoded[tuple(r["source_ids"])] for r in groups["train"]]),
        callbacks=[PilotChecks()])
    trainer.model_accepts_loss_kwargs = False
    # Recompute an untrained baseline in this environment; retain old outputs for comparison.
    with model.disable_adapter():
        snapshot(0, 0)
    trainer.train()
    changed = any(not torch.equal(before[name], value.detach().cpu()) for name, value in parameters.items())
    validate_snapshots(report)
    if (trainer.state.global_step != TOTAL_STEPS or len(status["gradient_checks"]) != TOTAL_STEPS
            or not changed):
        raise RuntimeError("Pilot completion checks failed")
    report.update(complete=True, adapter_weights_changed=True, optimizer_steps_completed=TOTAL_STEPS,
                  original_validation_loss=report["snapshots"][0]["validation_loss"],
                  fine_tuned_validation_loss=report["snapshots"][-1]["validation_loss"])
    persist()
    (output_dir / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2)+"\n")
    status.update(phase="complete", complete=True, adapter_weights_changed=True,
                  optimizer_steps_completed=TOTAL_STEPS)
    save_status()
    print("Pilot complete: four epochs, 96 updates, five snapshots of all 12 validation examples", flush=True)


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", required=True)
    parser.add_argument("--baseline", required=True)
    parser.add_argument("--smoke-status", required=True)
    parser.add_argument("--output-dir", default="/kaggle/working/reviewed_4b_pilot")
    parser.add_argument("--check", action="store_true")
    args = parser.parse_args(argv)
    groups, baseline = load_inputs(args.dataset, args.baseline)
    validate_smoke(args.smoke_status, args.dataset)
    print("Validated 48 train / 12 validation / 12 reserved test rows, frozen prompts, and successful smoke gate")
    if args.check:
        print("Dry-run OK. Four pilot epochs planned; no model downloaded or trained.")
        return
    output_dir = Path(args.output_dir)
    if output_dir.exists() and any(output_dir.iterdir()):
        raise ValueError("Output directory must be empty; use a fresh run to avoid mixing epochs")
    try:
        run_training(groups, baseline, args.dataset, output_dir)
    except Exception as exc:
        for name in ("pilot_status.json", "evaluation.json"):
            path = output_dir / name
            if path.exists():
                report = json.loads(path.read_text())
                report.update(complete=False, error_type=type(exc).__name__)
                path.write_text(json.dumps(report, indent=2, ensure_ascii=False)+"\n", encoding="utf-8")
        raise


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path
import subprocess
import sys

files = list(Path("/kaggle/input").rglob("fama_jensen_1983_pilot_reviewed.jsonl"))
baselines = list(Path("/kaggle/input").rglob("gemma-3-4b-it.json"))
smokes = list(Path("/kaggle/input").rglob("gemma4b_reviewed_smoke/smoke_status.json"))
if len(files) != 1 or len(baselines) != 1 or len(smokes) != 1:
    raise FileNotFoundError(f"Attach one reviewed Dataset, saved 4B baseline, and successful smoke output; found {len(files)}, {len(baselines)}, {len(smokes)}")
command = [sys.executable, "/kaggle/working/track_reviewed_4b.py", "--dataset", str(files[0]), "--baseline", str(baselines[0]), "--smoke-status", str(smokes[0])]
subprocess.run(command + ["--check"], check=True)
subprocess.run(command, check=True)
print("Review /kaggle/working/reviewed_4b_pilot/evaluation.json and claim_review.md")
